# 🎯 YOLO26 — Detección, segmentación, clasificación, pose y OBB

### Visión por Computador · Universidad de Santiago de Chile
**Diego Fischer** — material adaptado para el curso, sobre la base del trabajo de Domingo Mery y de la documentación oficial de Ultralytics.

---

[YOLO](https://docs.ultralytics.com/) (*You Only Look Once*) es una familia de modelos de visión en tiempo real. A diferencia de los detectores en dos etapas, YOLO resuelve la detección de objetos en una sola pasada de la red, lo que le da su característica velocidad. En este cuaderno usamos **YOLO26** a través de la librería `ultralytics`, que unifica bajo una misma API las cinco tareas más comunes de visión:

| Tarea | Sufijo | Qué entrega |
|-------|--------|-------------|
| Detección | *(sin sufijo)* | cajas + clase |
| Segmentación | `-seg` | máscara por objeto |
| Clasificación | `-cls` | etiqueta de la imagen completa |
| Pose | `-pose` | 17 puntos clave del cuerpo |
| OBB | `-obb` | cajas orientadas (con ángulo) |

> 💡 Antes de ejecutar, en Colab activa la GPU: **Entorno de ejecución → Cambiar tipo de entorno → GPU**.


## 0. Preparación del entorno

Instalamos `ultralytics` (trae consigo PyTorch y todas las dependencias) y corremos `checks()`, que reporta versiones, RAM, disco y si la GPU está disponible. Si `checks()` no muestra una GPU, revisa la configuración del entorno de ejecución.

In [ ]:
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

### Función auxiliar: dibujar el esqueleto

Más adelante, en la sección de **Pose**, vamos a querer dibujar nosotros mismos el esqueleto sobre la imagen en lugar de usar el render por defecto de Ultralytics. La siguiente función recibe una imagen y el tensor de *keypoints* (los 17 puntos del formato COCO) y traza las conexiones entre articulaciones en rojo, con un punto verde sobre cada articulación. Ignora los puntos en `(0, 0)`, que es como YOLO marca los keypoints no detectados.

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

def dibujar_esqueleto(imagen, keypoints):

  if imagen is None:
      print("Error: No se pudo cargar la imagen")
  else:
      image_to_draw = imagen.copy()

      # Convertir el tensor de keypoints a un arreglo numpy de forma (N, 2)
      points = keypoints.squeeze().cpu().numpy()

      # Función auxiliar: un punto es válido si no es (0, 0)
      def is_valid(pt):
          return pt[0] != 0 or pt[1] != 0

      # Formato de las líneas
      line_color = (0, 0, 255)  # Líneas rojas (BGR)
      thickness = 2

      # --- Conexiones entre articulaciones ---
      # Cada par (origen, destino) son índices de keypoints a unir
      connections = [
          # 0 con 1, 2, 3, 4
          (0, 1), (0, 2), (1, 2), (1, 3), (2, 4),
          (3, 5), (4, 6),
          # 5 con 6 y 7
          (5, 6), (5, 7),
          # 7 con 9
          (7, 9),
          # 6 con 8 y 8 con 10
          (6, 8), (8, 10),
          # tronco: hombros con caderas
          (5, 11), (6, 12),
          (11, 12), (11, 13),
          # 13 con 15
          (13, 15),
          # 12 con 14
          (12, 14),
          # 14 con 16
          (14, 16)
      ]

      # Dibujar las conexiones
      for start_idx, end_idx in connections:
          if start_idx < len(points) and end_idx < len(points):
              pt1 = points[start_idx]
              pt2 = points[end_idx]
              if is_valid(pt1) and is_valid(pt2):
                  cv2.line(image_to_draw, (int(pt1[0]), int(pt1[1])), (int(pt2[0]), int(pt2[1])), line_color, thickness)

      # --- Dibujar un punto sobre cada articulación, encima de las líneas ---
      for point in points:
          if is_valid(point):
              cv2.circle(image_to_draw, (int(point[0]), int(point[1])), 5, (0, 255, 0), -1)

      # Mostrar el resultado
      cv2_imshow(image_to_draw)

## 1. Predicción y entrenamiento desde la línea de comandos (CLI)

YOLO26 se puede usar directamente desde la línea de comandos con el comando `yolo`, que acepta distintas tareas y modos, además de argumentos adicionales como `imgsz=640`. Puedes ver la lista completa de [argumentos](https://docs.ultralytics.com/usage/cfg/) y más detalles en la [documentación de predicción](https://docs.ultralytics.com/modes/predict/).

En la celda siguiente corremos una predicción sobre una imagen propia (alojada en Dropbox). El resultado anotado se guarda en `runs/detect/predict/`.

In [ ]:
# Inferencia sobre una imagen con YOLO26n

!yolo predict model=yolo26n.pt source='https://www.dropbox.com/scl/fi/3774x2p4s1ewzdh0dyamj/Diego-2.jpeg?rlkey=vceph8h28halnv2pyzp5rt23e'

## 2. Validación

Validamos la precisión de un modelo sobre los conjuntos `val` o `test` del dataset [COCO](https://docs.ultralytics.com/datasets/detect/coco/). Los [modelos de YOLO26](https://github.com/ultralytics/ultralytics#models) se descargan automáticamente la primera vez que se usan. Más información en la [documentación de validación](https://docs.ultralytics.com/modes/val/).

In [ ]:
# Descargar el set de validación de COCO
from ultralytics.utils.downloads import download

download('https://ultralytics.com/assets/coco2017val.zip', unzip=True, dir='datasets')  # ~780 MB, 5000 imágenes

In [ ]:
# Validar YOLO26n sobre COCO8
!yolo val model=yolo26n.pt data=coco8.yaml

## 3. Entrenamiento

Se puede entrenar YOLO26 sobre datasets de [detección](https://docs.ultralytics.com/tasks/detect/), [segmentación](https://docs.ultralytics.com/tasks/segment/), [semántica](https://docs.ultralytics.com/tasks/semantic/), [clasificación](https://docs.ultralytics.com/tasks/classify/) y [pose](https://docs.ultralytics.com/tasks/pose/). Más información en la [documentación de entrenamiento](https://docs.ultralytics.com/modes/train/).

`coco8` es un dataset minúsculo (8 imágenes) pensado justamente para probar que la tubería de entrenamiento funciona sin esperar horas.

In [ ]:
# Entrenar YOLO26n sobre COCO8 durante 3 épocas
!yolo train model=yolo26n.pt data=coco8.yaml epochs=3 imgsz=640

## 4. Exportación

Exporta un modelo YOLO a cualquiera de los formatos soportados con el argumento `format`, por ejemplo `format=onnx`. Más información en la [documentación de exportación](https://docs.ultralytics.com/modes/export/).

- 💡 Tip: exporta a [ONNX](https://docs.ultralytics.com/integrations/onnx/) o [OpenVINO](https://docs.ultralytics.com/integrations/openvino/) para acelerar hasta 3× en CPU.
- 💡 Tip: exporta a [TensorRT](https://docs.ultralytics.com/integrations/tensorrt/) para acelerar hasta 5× en GPU.

| Formato | Argumento `format` | Modelo | Metadatos | Argumentos |
|--------|-----------------|-------|----------|------------|
| [PyTorch](https://pytorch.org/) | - | `yolo26n.pt` | ✅ | - |
| [TorchScript](https://docs.ultralytics.com/integrations/torchscript) | `torchscript` | `yolo26n.torchscript` | ✅ | `imgsz`, `batch`, `dynamic`, `optimize`, `quantize`, `nms`, `device` |
| [ONNX](https://docs.ultralytics.com/integrations/onnx) | `onnx` | `yolo26n.onnx` | ✅ | `imgsz`, `batch`, `data`, `fraction`, `dynamic`, `quantize`, `opset`, `simplify`, `nms`, `device` |
| [OpenVINO](https://docs.ultralytics.com/integrations/openvino) | `openvino` | `yolo26n_openvino_model/` | ✅ | `imgsz`, `batch`, `data`, `fraction`, `dynamic`, `quantize`, `nms`, `device` |
| [TensorRT](https://docs.ultralytics.com/integrations/tensorrt) | `engine` | `yolo26n.engine` | ✅ | `imgsz`, `batch`, `data`, `fraction`, `dynamic`, `quantize`, `simplify`, `nms`, `device`, `workspace` |
| [CoreML](https://docs.ultralytics.com/integrations/coreml) | `coreml` | `yolo26n.mlpackage` | ✅ | `imgsz`, `batch`, `quantize`, `nms`, `device` |
| [TF SavedModel](https://docs.ultralytics.com/integrations/tf-savedmodel) | `saved_model` | `yolo26n_saved_model/` | ✅ | `imgsz`, `batch`, `data`, `fraction`, `quantize`, `keras`, `nms`, `device` |
| [TF GraphDef](https://docs.ultralytics.com/integrations/tf-graphdef) | `pb` | `yolo26n.pb` | ❌ | `imgsz`, `batch`, `device` |
| [TF Edge TPU](https://docs.ultralytics.com/integrations/edge-tpu) | `edgetpu` | `yolo26n_edgetpu.tflite` | ✅ | `imgsz`, `quantize`, `data`, `fraction`, `device` |
| [PaddlePaddle](https://docs.ultralytics.com/integrations/paddlepaddle) | `paddle` | `yolo26n_paddle_model/` | ✅ | `imgsz`, `batch`, `device` |
| [MNN](https://docs.ultralytics.com/integrations/mnn) | `mnn` | `yolo26n.mnn` | ✅ | `imgsz`, `batch`, `quantize`, `device` |
| [NCNN](https://docs.ultralytics.com/integrations/ncnn) | `ncnn` | `yolo26n_ncnn_model/` | ✅ | `imgsz`, `batch`, `quantize`, `device` |
| [IMX500](https://docs.ultralytics.com/integrations/sony-imx500) | `imx` | `yolo26n_imx_model/` | ✅ | `imgsz`, `quantize`, `data`, `fraction`, `device` |
| [RKNN](https://docs.ultralytics.com/integrations/rockchip-rknn) | `rknn` | `yolo26n_rknn_model/` | ✅ | `imgsz`, `batch`, `name`, `quantize`, `data`, `fraction`, `device` |
| [ExecuTorch](https://docs.ultralytics.com/integrations/executorch) | `executorch` | `yolo26n_executorch_model/` | ✅ | `imgsz`, `device` |
| [Axelera AI](https://docs.ultralytics.com/integrations/axelera) | `axelera` | `yolo26n_axelera_model/` | ✅ | `imgsz`, `quantize`, `data`, `fraction`, `device` |
| [DEEPX](https://docs.ultralytics.com/integrations/deepx) | `deepx` | `yolo26n_deepx_model/` | ✅ | `imgsz`, `quantize`, `data`, `optimize`, `device` |
| [Qualcomm QNN](https://docs.ultralytics.com/integrations/qnn) | `qnn` | `yolo26n_qnn.onnx` | ✅ | `imgsz`, `batch`, `name`, `quantize`, `data`, `fraction`, `device` |
| [LiteRT](https://docs.ultralytics.com/integrations/litert) | `litert` | `yolo26n.tflite` | ✅ | `imgsz`, `quantize`, `batch`, `data`, `fraction`, `device` |

In [ ]:
!yolo export model=yolo26n.pt format=torchscript

## 5. Uso desde Python

YOLO26 fue diseñado con una filosofía *Python-first*. Los modelos se pueden cargar desde un checkpoint entrenado o crearse desde cero, y luego se usan los métodos `train`, `val`, `predict` y `export`. Ejemplos detallados en la [documentación de Python](https://docs.ultralytics.com/usage/python/).

In [ ]:
from ultralytics import YOLO

# Cargar un modelo
model = YOLO('yolo26n.yaml')  # construir un modelo nuevo desde cero
model = YOLO('yolo26n.pt')    # cargar un modelo preentrenado (recomendado para entrenar)

# Usar el modelo
results = model.train(data='coco8.yaml', epochs=3)  # entrenar
results = model.val()                                # evaluar sobre el set de validación
results = model('https://ultralytics.com/images/bus.jpg')  # predecir sobre una imagen
results = model.export(format='onnx')                # exportar a formato ONNX

### Más opciones de entrenamiento

El método `train` acepta argumentos para distribuir el entrenamiento en varias GPU o para limitar su duración por tiempo en lugar de por épocas.

In [ ]:
# Entrenar con 2 GPU
results = model.train(data="coco8.yaml", epochs=100, imgsz=640, device=[0, 1])

# Entrenar usando la GPU más desocupada
results = model.train(data="coco8.yaml", epochs=100, imgsz=640, device=-1)

# Entrenar usando las dos GPU más desocupadas
results = model.train(data="coco8.yaml", epochs=100, imgsz=640, device=[-1, -1])

# Limitar el entrenamiento a un máximo de 2.5 horas
results = model.train(data="coco8.yaml", epochs=100, imgsz=640, time=2.5)

### Retomar un entrenamiento interrumpido

Si un entrenamiento se corta (por ejemplo, porque Colab cerró la sesión), se puede reanudar desde el último checkpoint con `resume=True`.

In [ ]:
model = YOLO("path/to/last.pt")  # cargar un modelo parcialmente entrenado

# Reanudar el entrenamiento
results = model.train(resume=True)

## 6. Las cinco tareas

YOLO26 puede entrenar, validar, predecir y exportar modelos para las tareas más comunes de visión: [detección](https://docs.ultralytics.com/tasks/detect/), [segmentación](https://docs.ultralytics.com/tasks/segment/), [semántica](https://docs.ultralytics.com/tasks/semantic/), [clasificación](https://docs.ultralytics.com/tasks/classify/) y [pose](https://docs.ultralytics.com/tasks/pose/). Más información en la [documentación de tareas](https://docs.ultralytics.com/tasks/).

<br><img width="1024" src="https://raw.githubusercontent.com/ultralytics/assets/main/im/banner-tasks.png">

### 6.1 Detección

Los modelos de **detección** de YOLO26 no llevan sufijo (son los modelos por defecto, p. ej. `yolo26n.pt`) y vienen preentrenados sobre COCO. Detalles completos en la [documentación de detección](https://docs.ultralytics.com/tasks/detect/).

In [ ]:
# Cargar YOLO26n, entrenarlo sobre COCO8 por 3 épocas y luego predecir
from ultralytics import YOLO

model_detect = YOLO('yolo26n.pt')             # modelo de detección preentrenado
model_detect.train(data='coco8.yaml', epochs=3)  # entrenar

In [ ]:
resultados = model_detect('https://ultralytics.com/images/bus.jpg')  # predecir sobre una imagen
for result in resultados:
    boxes = result.boxes      # objeto Boxes con las cajas detectadas
    result.show()             # mostrar en pantalla
    result.save(filename="result.jpg")  # guardar en disco

### 6.2 Segmentación

Los modelos de **segmentación** usan el sufijo `-seg` (p. ej. `yolo26n-seg.pt`) y vienen preentrenados sobre COCO. Además de la caja, entregan una máscara por cada objeto. Detalles completos en la [documentación de segmentación](https://docs.ultralytics.com/tasks/segment/).

In [ ]:
# Cargar YOLO26n-seg, entrenarlo sobre COCO8-seg por 3 épocas y predecir
from ultralytics import YOLO

model_seg = YOLO('yolo26n-seg.pt')                  # modelo de segmentación preentrenado
model_seg.train(data='coco8-seg.yaml', epochs=3)    # entrenar

In [ ]:

resultados = model_seg('https://ultralytics.com/images/bus.jpg')  # predecir
for result in resultados:
    boxes = result.boxes      # cajas detectadas
    masks = result.masks      # máscaras de segmentación
    result.show()             # mostrar en pantalla
    result.save(filename="result.jpg")  # guardar en disco

### 6.3 Clasificación

Los modelos de **clasificación** usan el sufijo `-cls` (p. ej. `yolo26n-cls.pt`) y vienen preentrenados sobre ImageNet. A diferencia de la detección, asignan una etiqueta a la imagen **completa**, sin localizar objetos. Detalles completos en la [documentación de clasificación](https://docs.ultralytics.com/tasks/classify/).

<details>
<summary>📋 Las 80 clases de COCO (clic para expandir)</summary>

```
0: person          16: dog            32: sports ball    48: sandwich       64: mouse
1: bicycle         17: horse          33: kite           49: orange         65: remote
2: car             18: sheep          34: baseball bat   50: broccoli       66: keyboard
3: motorcycle      19: cow            35: baseball glove 51: carrot         67: cell phone
4: airplane        20: elephant       36: skateboard     52: hot dog        68: microwave
5: bus             21: bear           37: surfboard      53: pizza          69: oven
6: train           22: zebra          38: tennis racket  54: donut          70: toaster
7: truck           23: giraffe        39: bottle         55: cake           71: sink
8: boat            24: backpack       40: wine glass     56: chair          72: refrigerator
9: traffic light   25: umbrella       41: cup            57: couch          73: book
10: fire hydrant   26: handbag        42: fork           58: potted plant   74: clock
11: stop sign      27: tie            43: knife          59: bed            75: vase
12: parking meter  28: suitcase       44: spoon          60: dining table   76: scissors
13: bench          29: frisbee        45: bowl           61: toilet         77: teddy bear
14: bird           30: skis           46: banana         62: tv             78: hair drier
15: cat            31: snowboard      47: apple          63: laptop         79: toothbrush
```
</details>

In [ ]:
# Cargar YOLO26n-cls y entrenarlo sobre MNIST160
from ultralytics import YOLO

model_clas = YOLO('yolo26n-cls.pt')              # modelo de clasificación preentrenado
model_clas.train(data='mnist160', epochs=40)     # entrenar

In [ ]:
resultados = model_clas('https://ultralytics.com/images/bus.jpg')  # predecir sobre una imagen
for result in resultados:
    top1 = result.probs.top1            # ID de la clase más probable
    top1_conf = result.probs.top1conf   # confianza de la predicción
    top1_name = result.names[top1]      # nombre de la clase más probable
    print(top1)
    print(top1_conf)
    print(top1_name)
    print(result.probs.top5)            # las 5 clases más probables

### 6.4 Pose

Los modelos de **pose** usan el sufijo `-pose` (p. ej. `yolo26n-pose.pt`) y vienen preentrenados sobre COCO Keypoints. El modelo por defecto estima **17 puntos clave**, cada uno asociado a una parte del cuerpo:

| Índice | Parte | Índice | Parte |
|--------|-------|--------|-------|
| 0 | Nariz | 9 | Muñeca izquierda |
| 1 | Ojo izquierdo | 10 | Muñeca derecha |
| 2 | Ojo derecho | 11 | Cadera izquierda |
| 3 | Oreja izquierda | 12 | Cadera derecha |
| 4 | Oreja derecha | 13 | Rodilla izquierda |
| 5 | Hombro izquierdo | 14 | Rodilla derecha |
| 6 | Hombro derecho | 15 | Tobillo izquierdo |
| 7 | Codo izquierdo | 16 | Tobillo derecho |
| 8 | Codo derecho | | |

Detalles completos en la [documentación de pose](https://docs.ultralytics.com/tasks/pose/).

In [ ]:
# Cargar YOLO26n-pose y entrenarlo sobre COCO8-pose por 3 épocas
from ultralytics import YOLO

model_pose = YOLO('yolo26n-pose.pt')                 # modelo de pose preentrenado
model_pose.train(data='coco8-pose.yaml', epochs=3)   # entrenar

Descargamos dos imágenes de prueba (judokas) para probar la estimación de pose con varias personas en escena.

In [ ]:
!wget -q -O judo_1.jpeg \
https://www.dropbox.com/scl/fi/lh8bg0oiiikckqjc4njo5/judo_1.jpeg?rlkey=pbn1nn82kbulbs9gtb2snkb2l&st=fv3k0ubr&dl=0
!wget -q -O judo_2.jpeg \
https://www.dropbox.com/scl/fi/1wwxt6sirz4lp59yci2eg/judo_2.jpeg?rlkey=ly6swh0ratj9xho8odlz3d00d&st=vxs4ovqc&dl=0

Primero, el render por defecto de Ultralytics sobre la primera imagen:

In [ ]:
resultados = model_pose('judo_1.jpeg')  # predecir sobre la imagen

for result in resultados:
    result.show()             # mostrar en pantalla
    result.save(filename="result.jpg")  # guardar en disco

Y ahora con nuestra propia función `dibujar_esqueleto`, que toma los keypoints de la primera persona detectada y traza el esqueleto a mano:

In [ ]:
base = cv2.imread('judo_1.jpeg')
keypoints = resultados[0].keypoints[0].xy
dibujar_esqueleto(base, keypoints)

Repetimos sobre la segunda imagen:

In [ ]:
resultados = model_pose('judo_2.jpeg')  # predecir sobre la imagen

for result in resultados:
    result.show()             # mostrar en pantalla
    result.save(filename="result.jpg")  # guardar en disco

In [ ]:
base = cv2.imread('judo_2.jpeg')
keypoints = resultados[0].keypoints[0].xy
dibujar_esqueleto(base, keypoints)

### 6.5 Cajas orientadas (OBB)

Los modelos de **OBB** (*Oriented Bounding Boxes*) usan el sufijo `-obb` (p. ej. `yolo26n-obb.pt`) y vienen preentrenados sobre el dataset DOTA. A diferencia de las cajas axis-aligned, las cajas orientadas incluyen un ángulo, lo que las hace ideales para objetos rotados como barcos, aviones o vehículos vistos desde arriba en imágenes aéreas. Detalles completos en la [documentación de OBB](https://docs.ultralytics.com/tasks/obb/).

In [ ]:
# Cargar YOLO26n-obb y entrenarlo sobre DOTA8 por 3 épocas
from ultralytics import YOLO

model_obb = YOLO('yolo26n-obb.pt')                # modelo OBB preentrenado
model_obb.train(data='dota8.yaml', epochs=3)      # entrenar

In [ ]:
resultados = model_obb('https://ultralytics.com/images/boats.jpg')
for result in resultados:           # predecir sobre una imagen
    obb = result.obb                # objeto con las cajas orientadas
    result.show()                   # mostrar en pantalla
    result.save(filename="resultado_OBB.jpg")  # guardar en disco

---

### Una nota sobre el uso responsable

La detección de personas y la estimación de pose son herramientas poderosas y de doble uso: las mismas técnicas que sirven para análisis deportivo, accesibilidad o investigación pueden emplearse en vigilancia masiva o seguimiento sin consentimiento. Al trabajar con imágenes de personas reales conviene tener presente el consentimiento, la privacidad y los sesgos de los datos de entrenamiento. Vale la pena discutir estos puntos en clase junto con la parte técnica.

---
*Cuaderno adaptado para el módulo de Visión por Computador del diplomado de inteligencia artificial aplicada a la industria tecnológica · Universidad de Santiago de Chile.*
---
<a href="https://example.com">07_YOLO26_Curso.ipynb</a> © 2026 by <a href="https://example.com">Diego Fischer</a> is licensed under <a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a><img src="https://mirrors.creativecommons.org/presskit/icons/cc.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/by.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;">